# LeetCode 4010 — Maximize Pair Strength Using GCD bằng Python

Link đề: https://leetcode.com/problems/maximize-pair-strength-using-gcd/ · Độ khó: **Easy** · Chủ đề: Array, Math, Number Theory, GCD, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def maxPairStrength(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **đúng một cặp chỉ số khác nhau** `i != j`; **sức mạnh** của cặp là `(nums[i] * nums[j]) / gcd(nums[i], nums[j])^2`; trả về sức mạnh **lớn nhất**.
2. Biết `math.gcd`, và nhận ra `a*b / g^2 = (a/g) * (b/g)` luôn là **số nguyên** (tích hai số nguyên tố cùng nhau).
3. Nắm kỹ thuật **vét cạn mọi cặp** `O(n^2)` (với `n <= 2000` vẫn đủ) và **cắt tỉa (pruning)** bằng cận trên `a*b` sau khi sắp xếp giảm dần.
4. Viết 4 cách (vét cạn, công thức `(a/g)*(b/g)`, loại trùng + sắp xếp, cắt tỉa) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: chọn hai số lớn nhất, quên chia cho `gcd^2`, dùng `/` (ra số thực), nhầm với `lcm`, bỏ qua trường hợp hai số bằng nhau.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy chọn **đúng một cặp** chỉ số `i != j`. **Sức mạnh** (strength) của cặp được định nghĩa:

`strength = (nums[i] * nums[j]) / gcd(nums[i], nums[j])^2`

với `gcd` là ước chung lớn nhất. Trả về sức mạnh **lớn nhất** có thể trên mọi cặp.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,3,5]` | `15` | Cặp `(3, 5)`: `3*5 / gcd(3,5)^2 = 15 / 1 = 15` |
| 2 | `[4,6,8]` | `12` | Cặp `(6, 8)`: `48 / gcd(6,8)^2 = 48 / 4 = 12` |
| 3 | `[3,3]` | `1` | Cặp `(3, 3)`: `9 / 3^2 = 1` |

Constraints: `2 <= len(nums) <= 2000`; `1 <= nums[i] <= 10^5`.

Chú ý: kết quả luôn là số nguyên, vì `a*b / g^2 = (a/g) * (b/g)` với `g = gcd(a, b)`.

In [1]:
from math import gcd

for nums in ([2, 3, 5], [4, 6, 8], [3, 3]):
    best = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            a, b = nums[i], nums[j]
            g = gcd(a, b)
            st = a * b // (g * g)
            print(f"  ({a},{b}): gcd={g}, strength={st}")
            best = max(best, st)
    print(nums, "->", best)

  (2,3): gcd=1, strength=6
  (2,5): gcd=1, strength=10
  (3,5): gcd=1, strength=15
[2, 3, 5] -> 15
  (4,6): gcd=2, strength=6
  (4,8): gcd=4, strength=2
  (6,8): gcd=2, strength=12
[4, 6, 8] -> 12
  (3,3): gcd=3, strength=1
[3, 3] -> 1


## 3. Kiến thức nền

### 3.1. GCD: `math.gcd`

`gcd(a, b)` là số nguyên dương lớn nhất chia hết cả `a` và `b`. Python có `math.gcd(a, b)` dùng thuật toán Euclid (`O(log min(a, b))`). `gcd(6, 8) = 2`, `gcd(3, 5) = 1`, `gcd(3, 3) = 3`.

In [2]:
from math import gcd
print(gcd(6, 8), gcd(3, 5), gcd(3, 3), gcd(12, 18))

2 1 3 6


### 3.2. `a // g` luôn chia hết, `a * b / g^2` là số nguyên

Vì `g` chia hết cả `a` và `b`, nên `a // g` và `b // g` là số nguyên. Khi đó `a*b / g^2 = (a // g) * (b // g)`: một tích hai số nguyên. Dùng `//` (chia nguyên) hoặc phép nhân sau khi chia từng số để tránh số thực.

In [3]:
a, b = 6, 8
g = gcd(a, b)
print(a // g, b // g, (a // g) * (b // g), a * b // (g * g), a * b / (g * g))

3 4 12 12 12.0


### 3.3. `/` khác `//`

`/` luôn trả về số thực (`float`), ví dụ `48 / 4 = 12.0`. LeetCode mong đợi kiểu `int`, nên dùng `//` khi biết chắc chia hết. Với số rất lớn, `/` còn có thể mất độ chính xác (nhưng ở đây `a*b <= 10^10` nên vẫn chính xác).

In [4]:
print(48 / 4, 48 // 4, type(48 / 4).__name__, type(48 // 4).__name__)

12.0 12 float int


### 3.4. Hai vòng lặp lồng nhau cho mọi cặp `(i, j)`, `i < j`

`for i in range(n): for j in range(i + 1, n):` đi qua mọi cặp **không thứ tự** đúng một lần: `n (n - 1) / 2` cặp. `strength` đối xứng theo `a, b` nên chỉ cần `i < j`. Với `n = 2000` có khoảng `2 * 10^6` cặp.

In [5]:
n = 2000
print(n * (n - 1) // 2)
print([(i, j) for i in range(4) for j in range(i + 1, 4)])

1999000
[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


### 3.5. `sorted(..., reverse=True)` và `set`

`sorted(nums, reverse=True)` sắp xếp giảm dần. `set(nums)` loại trùng. Hai công cụ này dùng cho cách cắt tỉa: duyệt các giá trị lớn trước, và nhớ rằng **hai số bằng nhau chỉ cho sức mạnh `1`**.

In [6]:
nums = [4, 6, 8, 6]
print(sorted(nums, reverse=True), sorted(set(nums), reverse=True))

[8, 6, 6, 4] [8, 6, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách nghĩ đầu tiên: **thử mọi cặp**, tính sức mạnh theo đúng công thức và giữ giá trị lớn nhất. Ràng buộc `n <= 2000` cho phép `O(n^2) = 2 * 10^6` cặp, mỗi cặp một `gcd`, vẫn nhanh trong Python (khoảng một giây hoặc ít hơn).

Hiểu công thức: gọi `g = gcd(a, b)`, `a = g * a'`, `b = g * b'` với `a', b'` nguyên tố cùng nhau. Khi đó `strength = (g a')(g b') / g^2 = a' * b'`. Sức mạnh là tích của hai phần **sau khi bỏ ước chung**; nên hai số **nguyên tố cùng nhau và lớn** cho sức mạnh lớn nhất (`gcd = 1` thì `strength = a * b`).

Tính tay `[6, 10, 15]`:

| cặp | `gcd` | `a'`, `b'` | sức mạnh |
|---|---|---|---|
| `(6, 10)` | 2 | 3, 5 | 15 |
| `(6, 15)` | 3 | 2, 5 | 10 |
| `(10, 15)` | 5 | 2, 3 | 6 |

Đáp án `15`. Lưu ý hai số **lớn nhất** `(10, 15)` chỉ cho `6`, nên chọn "hai số lớn nhất" là **sai**.

In [7]:
nums = [6, 10, 15]
for i in range(3):
    for j in range(i + 1, 3):
        a, b = nums[i], nums[j]
        g = gcd(a, b)
        print((a, b), "gcd =", g, "->", a // g, "*", b // g, "=", (a // g) * (b // g))

(6, 10) gcd = 2 -> 3 * 5 = 15
(6, 15) gcd = 3 -> 2 * 5 = 10
(10, 15) gcd = 5 -> 2 * 3 = 6


## 5. Cách 1 — Brute Force / baseline: mọi cặp với công thức gốc

Với mọi `i < j`: `g = gcd(a, b)`, `strength = a * b // (g * g)`, giữ giá trị lớn nhất. Đây chính xác là đề, `O(n^2 log V)` với `V = 10^5`.

"Nút thắt": với `n = 2000` vẫn chấp nhận được, nhưng nếu `n` lớn hơn (như `10^5`) thì không. Phần sau sẽ giảm số cặp cần xét bằng **cận trên**.

In [8]:
from math import gcd

class SolutionBrute:
    def maxPairStrength(self, nums):
        best = 0
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                g = gcd(nums[i], nums[j])
                s = nums[i] * nums[j] // (g * g)
                if s > best:
                    best = s
        return best

print(SolutionBrute().maxPairStrength([2, 3, 5]), SolutionBrute().maxPairStrength([4, 6, 8]), SolutionBrute().maxPairStrength([3, 3]))

15 12 1


**Độ phức tạp:** Time `O(n^2 * log V)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 (cận trên):** `strength(a, b) = a' * b' <= a * b` (vì `a' <= a`, `b' <= b`), dấu bằng khi `gcd = 1`. Nên `a * b` là **cận trên** của sức mạnh cặp `(a, b)`.

**Quan sát 2 (sắp xếp giảm dần + cắt tỉa):** sắp xếp `nums` giảm dần `v_0 >= v_1 >= ...`. Với số ở vị trí `i`, mọi đối tác `j > i` có `v_j <= v_{i+1}` nên sức mạnh `<= v_i * v_{i+1}`.

- Nếu `v_i * v_{i+1} <= best` thì **không cặp nào bắt đầu từ `i` trở đi** có thể vượt `best` (vì `v_k * v_{k+1}` giảm dần theo `k`): dừng vòng ngoài (`break`).
- Tương tự trong vòng trong: khi `v_i * v_j <= best` thì mọi `j` lớn hơn (giá trị nhỏ hơn) cũng không vượt: dừng vòng trong.

**Quan sát 3 (trùng lặp):** hai giá trị **bằng nhau** `x, x` cho sức mạnh `1`, luôn là mức tối thiểu có thể (mỗi cặp có `strength >= 1`). Nên có thể **loại trùng** (`set`) khi `len(set) >= 2`; nếu mọi phần tử bằng nhau thì đáp án là `1`.

**Invariant:** `best` luôn là sức mạnh lớn nhất trong các cặp đã xét; cắt tỉa chỉ bỏ những cặp có cận trên `<= best`, nên không bỏ sót đáp án.

## 7. Các cách tiếp theo

### Cách 2 — Viết theo `(a // g) * (b // g)` và `max` với generator

Cùng độ phức tạp nhưng ngắn gọn, tránh nhân lớn rồi mới chia.

In [9]:
class SolutionCoprime:
    def maxPairStrength(self, nums):
        best = 0
        for i, a in enumerate(nums):
            for b in nums[i + 1:]:
                g = gcd(a, b)
                best = max(best, (a // g) * (b // g))
        return best

print(SolutionCoprime().maxPairStrength([2, 3, 5]), SolutionCoprime().maxPairStrength([4, 6, 8]), SolutionCoprime().maxPairStrength([3, 3]))

15 12 1


**Độ phức tạp:** Time `O(n^2 log V)`, Space `O(n)` do lát cắt `nums[i + 1:]` (có thể `O(1)` dùng chỉ số).

### Cách 3 — Loại trùng rồi vét cạn trên các giá trị khác nhau

Chỉ cần xét các **giá trị khác nhau**: cặp hai giá trị bằng nhau cho `1`. Nếu chỉ có một giá trị khác nhau thì đáp án `1`. Giảm số cặp nếu có nhiều số trùng.

In [10]:
class SolutionDedup:
    def maxPairStrength(self, nums):
        vals = sorted(set(nums))
        best = 1                                   # cặp hai số bằng nhau (hoặc bất kỳ cặp nào) cho ít nhất 1
        for i in range(len(vals)):
            for j in range(i + 1, len(vals)):
                g = gcd(vals[i], vals[j])
                best = max(best, (vals[i] // g) * (vals[j] // g))
        return best

print(SolutionDedup().maxPairStrength([2, 3, 5]), SolutionDedup().maxPairStrength([4, 6, 8]), SolutionDedup().maxPairStrength([3, 3]))

15 12 1


**Độ phức tạp:** Time `O(d^2 log V + n log n)` với `d` số giá trị khác nhau, Space `O(d)`.

### Cách 4 — Sắp xếp giảm dần + cắt tỉa bằng cận trên `a * b`

Dùng quan sát 2: duyệt `i` từ giá trị lớn nhất; `break` khi cận trên không thể vượt `best`.

In [11]:
class SolutionPrune:
    def maxPairStrength(self, nums):
        v = sorted(nums, reverse=True)
        n = len(v)
        best = 1
        for i in range(n - 1):
            if v[i] * v[i + 1] <= best:            # mọi cặp bắt đầu từ i trở đi có cận trên <= best
                break
            for j in range(i + 1, n):
                if v[i] * v[j] <= best:            # các j sau còn nhỏ hơn nữa
                    break
                g = gcd(v[i], v[j])
                s = (v[i] // g) * (v[j] // g)
                if s > best:
                    best = s
        return best

print(SolutionPrune().maxPairStrength([2, 3, 5]), SolutionPrune().maxPairStrength([4, 6, 8]), SolutionPrune().maxPairStrength([3, 3]))

15 12 1


**Độ phức tạp:** Time trường hợp xấu `O(n^2 log V)` (khi cận trên không cắt được), thường **gần `O(n log V)`** trên dữ liệu ngẫu nhiên vì tìm được cặp nguyên tố cùng nhau lớn rất sớm. Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 (cắt tỉa) với `nums = [6, 10, 15]`, sắp giảm dần `v = [15, 10, 6]`, `best = 1`:

| `i` | `j` | `v[i]*v[j]` | so với `best` | `gcd` | sức mạnh | `best` |
|---|---|---|---|---|---|---|
| 0 | 1 | 150 | `150 > 1` | 5 | `3*2 = 6` | 6 |
| 0 | 2 | 90 | `90 > 6` | 3 | `5*2 = 10` | 10 |
| 1 | 2 | 60 | `60 > 10` | 2 | `5*3 = 15` | 15 |

Kết quả `15` (khớp phân tích mục 4).

In [12]:
def trace_prune(nums):
    v = sorted(nums, reverse=True)
    n, best = len(v), 1
    print("v =", v, "best =", best)
    for i in range(n - 1):
        if v[i] * v[i + 1] <= best:
            print(f"  i={i}: v[i]*v[i+1]={v[i] * v[i + 1]} <= best={best} -> dừng vòng ngoài")
            break
        for j in range(i + 1, n):
            if v[i] * v[j] <= best:
                print(f"  (i={i}, j={j}): cận trên {v[i] * v[j]} <= best={best} -> dừng vòng trong")
                break
            g = gcd(v[i], v[j])
            s = (v[i] // g) * (v[j] // g)
            best = max(best, s)
            print(f"  (i={i}, j={j}): {v[i]},{v[j]} gcd={g} -> sức mạnh {s}, best={best}")
    return best

assert trace_prune([6, 10, 15]) == 15
assert trace_prune([4, 6, 8]) == 12
assert trace_prune([3, 3]) == 1

v = [15, 10, 6] best = 1
  (i=0, j=1): 15,10 gcd=5 -> sức mạnh 6, best=6
  (i=0, j=2): 15,6 gcd=3 -> sức mạnh 10, best=10
  (i=1, j=2): 10,6 gcd=2 -> sức mạnh 15, best=15
v = [8, 6, 4] best = 1
  (i=0, j=1): 8,6 gcd=2 -> sức mạnh 12, best=12
  (i=0, j=2): 8,4 gcd=4 -> sức mạnh 2, best=12
  (i=1, j=2): 6,4 gcd=2 -> sức mạnh 6, best=12
v = [3, 3] best = 1
  (i=0, j=1): 3,3 gcd=3 -> sức mạnh 1, best=1


In [13]:
def trace_formula(a, b):
    g = gcd(a, b)
    print(f"a={a}, b={b}: g={g}; a*b/g^2 = {a * b}/{g * g} = {a * b // (g * g)}; (a/g)*(b/g) = {a // g}*{b // g} = {(a // g) * (b // g)}")
    assert a * b % (g * g) == 0
    return (a // g) * (b // g)

assert trace_formula(6, 8) == 12 and trace_formula(3, 5) == 15 and trace_formula(3, 3) == 1

a=6, b=8: g=2; a*b/g^2 = 48/4 = 12; (a/g)*(b/g) = 3*4 = 12
a=3, b=5: g=1; a*b/g^2 = 15/1 = 15; (a/g)*(b/g) = 3*5 = 15
a=3, b=3: g=3; a*b/g^2 = 9/9 = 1; (a/g)*(b/g) = 1*1 = 1


## 9. Code nộp LeetCode

Dùng Cách 4 (cắt tỉa), vẫn đúng trong trường hợp xấu (chỉ chậm hơn một chút) và nhanh trên dữ liệu thường gặp.

In [14]:
from math import gcd
from typing import List, Optional


class Solution:
    def maxPairStrength(self, nums: List[int]) -> int:
        v = sorted(nums, reverse=True)
        n = len(v)
        best = 1
        for i in range(n - 1):
            if v[i] * v[i + 1] <= best:
                break
            for j in range(i + 1, n):
                if v[i] * v[j] <= best:
                    break
                g = gcd(v[i], v[j])
                s = (v[i] // g) * (v[j] // g)
                if s > best:
                    best = s
        return best


print(Solution().maxPairStrength([2, 3, 5]), Solution().maxPairStrength([4, 6, 8]), Solution().maxPairStrength([3, 3]))

15 12 1


## 10. Vì sao thuật toán đúng

1. **Công thức.** Với `g = gcd(a, b)`, `a = g a'`, `b = g b'`: `a*b / g^2 = a' b'`, là số nguyên. Mọi cách đều tính đại lượng này (Cách 1 qua `a*b // g^2`, Cách 2, 3, 4 qua `(a//g)*(b//g)`, hai biểu thức bằng nhau).
2. **Cận dưới 1.** Mỗi cặp có `a', b' >= 1` nên `strength >= 1`. Do đó khởi tạo `best = 1` là an toàn khi `n >= 2` (luôn có ít nhất một cặp).
3. **Cận trên.** `a' <= a`, `b' <= b` nên `strength <= a*b`.
4. **Cắt tỉa vòng trong.** Với `v` giảm dần và `j' > j`: `v[i] * v[j'] <= v[i] * v[j]`. Nếu `v[i]*v[j] <= best` thì mọi `j' > j` cũng cho cận trên `<= best`, không thể làm `best` tăng. Bỏ qua an toàn.
5. **Cắt tỉa vòng ngoài.** Với `i' > i` và `j > i'`: `v[i'] * v[j] <= v[i+1] * v[i+2] <= v[i] * v[i+1]` (vì `v` giảm dần: `v[i'] <= v[i+1]` và `v[j] <= v[i+1]`... chính xác hơn `v[i'] * v[j] <= v[i'] * v[i'+1] <= v[i] * v[i+1]`). Vậy nếu `v[i]*v[i+1] <= best` thì mọi cặp còn lại không vượt `best`.
6. Cách 3 loại các cặp trùng giá trị (cho `1`) và Cách 1 vét cạn mọi cặp: cùng đáp án.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) <= 2000`, `V = 10^5`.

- Cách 1: Time `O(n^2 log V)` (khoảng `2 * 10^6` lần `gcd`), Space `O(1)`.
- Cách 2: như Cách 1, Space `O(n)` vì lát cắt.
- Cách 3: Time `O(d^2 log V)` với `d <= n` số giá trị khác nhau.
- Cách 4 (nộp): Time trường hợp xấu `O(n^2 log V)`; thực tế thường nhanh hơn nhiều nhờ cắt tỉa. Space `O(n)` để sắp xếp.

Ghi nhớ: với `n <= 2000`, `O(n^2)` vét cạn đã đủ để qua; cắt tỉa là phần thưởng.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random, time

solutions = [SolutionBrute(), SolutionCoprime(), SolutionDedup(), SolutionPrune(), Solution()]

fixed = [
    ([2, 3, 5], 15), ([4, 6, 8], 12), ([3, 3], 1), ([1, 1], 1), ([1, 100000], 100000),
    ([6, 10, 15], 15), ([2, 4], 2), ([100000, 99999], 100000 * 99999), ([12, 18], 6),
    ([7, 7, 7, 7], 1), ([2, 2, 3], 6), ([10, 20, 30, 40], 12),
]
for nums, expected in fixed:
    for sol in solutions:
        assert sol.maxPairStrength(list(nums)) == expected, (type(sol).__name__, nums)
print("test cố định OK:", len(fixed), "ca")

random.seed(4010)
for _ in range(400):
    n = random.randint(2, 60)
    hi = random.choice([5, 30, 1000, 10 ** 5])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = SolutionBrute().maxPairStrength(list(nums))
    for sol in solutions:
        assert sol.maxPairStrength(list(nums)) == expected, (type(sol).__name__, nums)
print("test ngẫu nhiên nhỏ OK")

# kiểm tra kích thước tối đa của đề: n = 2000
nums = [random.randint(1, 10 ** 5) for _ in range(2000)]
t0 = time.time(); a = SolutionBrute().maxPairStrength(nums); t1 = time.time()
b = Solution().maxPairStrength(nums); t2 = time.time()
assert a == b
print(f"n=2000: kết quả={a}; vét cạn {t1 - t0:.2f}s, cắt tỉa {t2 - t1:.4f}s")

test cố định OK: 12 ca
test ngẫu nhiên nhỏ OK


n=2000: kết quả=9996700260; vét cạn 0.26s, cắt tỉa 0.0002s


### Test thuộc tính (property test)

(1) Đáp án `>= 1` và `<= tích hai số lớn nhất`. (2) Hoán vị `nums` không đổi kết quả. (3) Nếu có hai số nguyên tố cùng nhau thì đáp án `>=` tích nhỏ nhất của một cặp như vậy. (4) Thêm một số bằng một phần tử có sẵn không làm giảm đáp án.

In [16]:
random.seed(40100)
for _ in range(300):
    n = random.randint(2, 40)
    nums = [random.randint(1, 500) for _ in range(n)]
    r = Solution().maxPairStrength(nums)
    top = sorted(nums, reverse=True)
    assert 1 <= r <= top[0] * top[1]
    sh = nums[:]; random.shuffle(sh)
    assert Solution().maxPairStrength(sh) == r
    assert Solution().maxPairStrength(nums + [random.choice(nums)]) >= r
    coprime = [nums[i] * nums[j] for i in range(n) for j in range(i + 1, n) if gcd(nums[i], nums[j]) == 1]
    if coprime:
        assert r >= max(coprime)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chọn hai số **lớn nhất**

Sức mạnh phụ thuộc `gcd`; hai số lớn nhất có thể chia sẻ ước chung lớn. `[6,10,15]`: hai số lớn nhất `(10, 15)` cho `6`, nhưng cặp `(6, 10)` cho `15`.

In [17]:
def wrong_two_largest(nums):
    a, b = sorted(nums)[-2:]
    g = gcd(a, b)
    return (a // g) * (b // g)                                  # SAI: bỏ qua các cặp khác

print(wrong_two_largest([6, 10, 15]), "!=", Solution().maxPairStrength([6, 10, 15]))
assert wrong_two_largest([6, 10, 15]) == 6 and Solution().maxPairStrength([6, 10, 15]) == 15

6 != 15


### Lỗi 2: quên chia cho `gcd^2` (chỉ lấy tích lớn nhất)

`a * b` chỉ bằng sức mạnh khi `gcd = 1`. `[4, 6, 8]`: tích lớn nhất `48`, sức mạnh đúng `12`.

In [18]:
def wrong_product(nums):
    return max(nums[i] * nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))    # SAI

print(wrong_product([4, 6, 8]), "!=", Solution().maxPairStrength([4, 6, 8]))
assert wrong_product([4, 6, 8]) == 48 and Solution().maxPairStrength([4, 6, 8]) == 12

48 != 12


### Lỗi 3: dùng `/` (ra số thực) thay vì `//`

`48 / 4 = 12.0` là `float`, không phải `int`; LeetCode có thể từ chối hoặc so sánh khác.

In [19]:
def wrong_float(nums):
    best = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            g = gcd(nums[i], nums[j])
            best = max(best, nums[i] * nums[j] / (g * g))        # SAI: / cho float
    return best

r = wrong_float([4, 6, 8])
print(r, type(r).__name__)
assert isinstance(r, float) and r == 12 and isinstance(Solution().maxPairStrength([4, 6, 8]), int)

12.0 float


### Lỗi 4: nhầm với `lcm`

`lcm(a, b) = a*b / gcd(a, b)` (chỉ chia `gcd` **một** lần). Sức mạnh chia `gcd` **hai** lần (`gcd^2`). `[4, 6, 8]`: `lcm(6, 8) = 24`, nhưng sức mạnh `12`.

In [20]:
def wrong_lcm(nums):
    return max(nums[i] * nums[j] // gcd(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))    # SAI

print(wrong_lcm([4, 6, 8]), "!=", Solution().maxPairStrength([4, 6, 8]))
assert wrong_lcm([4, 6, 8]) == 24 and Solution().maxPairStrength([4, 6, 8]) == 12

24 != 12


### Lỗi 5: cắt tỉa bằng `<` thay vì `<=`, hoặc cắt tỉa theo thứ tự tăng dần

Cắt tỉa chỉ đúng khi duyệt **giảm dần** (để cận trên giảm theo `j`). Nếu sắp xếp **tăng dần** rồi `break` khi `a*b <= best`, ta bỏ nhầm cặp tốt phía sau (giá trị lớn hơn).

In [21]:
def wrong_prune_ascending(nums):
    v = sorted(nums)                                              # SAI: tăng dần
    n, best = len(v), 1
    for i in range(n - 1):
        for j in range(i + 1, n):
            if v[i] * v[j] <= best:
                break                                             # cận trên tăng theo j: break là sai
            g = gcd(v[i], v[j])
            best = max(best, (v[i] // g) * (v[j] // g))
    return best

print(wrong_prune_ascending([2, 3, 5, 7]), "vs", Solution().maxPairStrength([2, 3, 5, 7]))
print(wrong_prune_ascending([1, 2, 2, 4]), "vs", Solution().maxPairStrength([1, 2, 2, 4]))
assert wrong_prune_ascending([1, 2, 2, 4]) == 2 and Solution().maxPairStrength([1, 2, 2, 4]) == 4
assert Solution().maxPairStrength([2, 3, 5, 7]) == 35

35 vs 35
2 vs 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đọc ràng buộc: `n <= 2000` -> `O(n^2)` vét cạn **chấp nhận được**; đừng đi tìm thuật toán thông minh quá sớm.
- `a*b / gcd^2 = (a/g)*(b/g)`: nhớ biến đổi để thấy số nguyên và cận trên `a*b`.
- Mẫu cắt tỉa: **sắp xếp giảm dần**, dùng cận trên `v[i]*v[j]` để `break`.
- Hai số **bằng nhau** cho sức mạnh `1` (tối thiểu); đáp án luôn `>= 1`.
- `math.gcd` có sẵn; `math.lcm` từ Python 3.9.
- Luôn thử các ca: hai phần tử, mọi phần tử bằng nhau, có số `1`, hai số nguyên tố cùng nhau lớn.

## 15. Biến thể của bài

1. **Greatest Common Divisor of Strings** (1071), **Find Greatest Common Divisor of Array** (1979): `gcd` cơ bản.
2. **Largest Component Size by Common Factor** (952, Hard): quan hệ `gcd > 1`.
3. **Number of Different Integers / GCD of Pairs** dạng "đếm cặp theo `gcd`" (ví dụ truy vấn `gcd` của mọi cặp): ý tưởng tính `gcd` trên cặp.
4. **Count Array Pairs Divisible by K** (2183): ghép cặp theo `gcd`.
5. Biến thể: tìm cặp có **`lcm` lớn nhất** hoặc `gcd` lớn nhất (`gcd` lớn nhất cần đếm bội số, `O(V log V)`).
6. Biến thể: `n` lên `10^5`: vét cạn không còn được; cần thuật toán theo **giá trị** (chỉ xét giá trị lớn nhất) hoặc ước/bội.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn công thức gốc | `a*b // gcd^2` mọi cặp | `O(n^2 log V)` | `O(1)` | Đúng chữ đề, trọng tài |
| 2. `(a//g)*(b//g)` | tránh nhân lớn | `O(n^2 log V)` | `O(1)`/`O(n)` | Gọn, cùng độ phức tạp |
| 3. Loại trùng | chỉ giá trị khác nhau | `O(d^2 log V)` | `O(d)` | Nhiều số trùng |
| 4. Sắp xếp giảm + cắt tỉa | `break` khi `a*b <= best` | tệ nhất `O(n^2 log V)`, thường rất nhanh | `O(n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().maxPairStrength([8, 12, 9])` và `Solution().maxPairStrength([5, 5, 5])` ra gì?

**Bài 2 (tính tay):** với `[14, 21, 35]`, tính sức mạnh ba cặp rồi đáp án.

**Bài 3 (điền TODO):** hoàn thành `min_pair_strength(nums)`: sức mạnh **nhỏ nhất** trên mọi cặp.

**Bài 4 (tự viết lại):** viết `best_pair(nums)` trả về `(i, j)` (chỉ số, `i < j`) đạt sức mạnh lớn nhất (cặp đầu tiên nếu có nhiều).

**Bài 5 (mini challenge):** viết `max_gcd_pair(nums)`: `gcd` lớn nhất trên mọi cặp (vét cạn) rồi so với cách đếm bội số cho `nums` lớn.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().maxPairStrength([8, 12, 9]), Solution().maxPairStrength([5, 5, 5]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def min_pair_strength_todo(nums):
    # TODO: vét cạn mọi cặp, lấy min của (a // g) * (b // g)
    return 0

72 1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `(8, 9)` nguyên tố cùng nhau. `[5,5,5]`: mọi cặp `(5, 5)`. Bài 2: `gcd(14, 21) = 7`, `gcd(14, 35) = 7`, `gcd(21, 35) = 7`.

**Mức 2 (vừa):** Bài 3: giống Cách 1 nhưng `min`. Bài 4: lưu `(best, i, j)`; dùng `>` để giữ cặp đầu. Bài 5: `max(gcd(a, b) ...)`.

**Mức 3 (gần đáp án):** Bài 1: `72` và `1`. Bài 2: `2*3 = 6`, `2*5 = 10`, `3*5 = 15` -> `15`. Bài 4: `best_pair([6, 10, 15]) = (0, 1)`. Bài 5: `max_gcd_pair([12, 18, 27]) = 9`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().maxPairStrength([8, 12, 9]) == 72 and Solution().maxPairStrength([5, 5, 5]) == 1
assert Solution().maxPairStrength([14, 21, 35]) == 15

# Bài 3
def min_pair_strength(nums):
    best = None
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            g = gcd(nums[i], nums[j])
            s = (nums[i] // g) * (nums[j] // g)
            best = s if best is None else min(best, s)
    return best

assert min_pair_strength([6, 10, 15]) == 6 and min_pair_strength([3, 3]) == 1

# Bài 4
def best_pair(nums):
    best, ans = 0, None
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            g = gcd(nums[i], nums[j])
            s = (nums[i] // g) * (nums[j] // g)
            if s > best:
                best, ans = s, (i, j)
    return ans

assert best_pair([6, 10, 15]) == (0, 1) and best_pair([3, 3]) == (0, 1)
i, j = best_pair([4, 6, 8])
assert (i, j) == (1, 2)

# Bài 5
def max_gcd_pair(nums):
    return max(gcd(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))

assert max_gcd_pair([12, 18, 27]) == 9 and max_gcd_pair([2, 3]) == 1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from math import gcd

class Solution:
    def maxPairStrength(self, nums: List[int]) -> int:
        v = sorted(nums, reverse=True)
        best = 1
        for i in range(len(v) - 1):
            if v[i] * v[i + 1] <= best:        # cận trên của mọi cặp còn lại
                break
            for j in range(i + 1, len(v)):
                if v[i] * v[j] <= best:
                    break
                g = gcd(v[i], v[j])
                best = max(best, (v[i] // g) * (v[j] // g))
        return best
```

Khung chung: "tối ưu theo cặp" với `n <= 2000` -> vét cạn `O(n^2)`; nếu muốn nhanh hơn, **sắp xếp + cận trên + break**.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Công thức đúng: `(a // g) * (b // g)` (chia `gcd` hai lần), trả `int` (không `/`).
- [ ] Cắt tỉa chỉ khi sắp xếp **giảm dần**; đã thử `[3, 3]`, `[6, 10, 15]` (hai số lớn nhất không tối ưu).